# Data Cleaning and Preparation
## Lab 2 — Customer Churn

**Dataset:** A simple Telecom Customer Churn dataset generated using `sklearn.datasets.make_classification`.

### Aim
To clean and prepare customer data for further churn analysis.

### Steps
1. Load the dataset.
2. Explore the data.
3. Handle missing values.
4. Remove duplicates.
5. Standardize text values.
6. Convert data types.
7. Handle simple outliers.
8. Create a new feature.
9. Normalize numerical data.
10. Split into training and testing data.
11. Export the cleaned dataset.


In [ ]:
import pandas as pd
import numpy as np

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

pd.set_option("display.max_columns", None)


## 1. Create and load the dataset

`make_classification()` is part of `sklearn.datasets`. It creates a simple classification dataset that we use here as a small telecom churn dataset.


In [ ]:
# Create a simple telecom customer dataset using sklearn
X, y = make_classification(
    n_samples=200,
    n_features=4,
    n_informative=3,
    n_redundant=0,
    random_state=42
)

df = pd.DataFrame(X, columns=[
    "Monthly_Charges",
    "Tenure_Months",
    "Support_Calls",
    "Data_Usage"
])

# Make values easier to understand
df["Monthly_Charges"] = (df["Monthly_Charges"] * 20 + 70).round(2)
df["Tenure_Months"] = (df["Tenure_Months"] * 10 + 30).round().astype(int)
df["Support_Calls"] = np.abs(df["Support_Calls"] * 2 + 3).round().astype(int)
df["Data_Usage"] = np.abs(df["Data_Usage"] * 5 + 10).round(2)

df["Contract"] = np.where(
    df["Tenure_Months"] > 36,
    "Long Term",
    "Month-to-Month"
)

df["Churn"] = np.where(y == 1, "Yes", "No")

# Add a few realistic data-quality problems
df.loc[5, "Monthly_Charges"] = np.nan
df.loc[15, "Contract"] = " month-to-month "
df.loc[25, "Contract"] = "LONG TERM"
df.loc[35, "Support_Calls"] = 100

# Add one duplicate row
df = pd.concat([df, df.iloc[[10]]], ignore_index=True)

df.to_csv("Telecom_Customer_Churn.csv", index=False)

print("Dataset created successfully.")
display(df.head())


## 2. Explore the dataset

In [ ]:
# Shape of the dataset
print("Rows and columns:", df.shape)

# Column names and data types
print("\nData types:")
print(df.dtypes)

# First few records
print("\nFirst 5 records:")
display(df.head())

# Basic statistics
print("\nBasic statistics:")
display(df.describe())


## 3. Check and handle missing values

In [ ]:
print("Missing values before cleaning:")
print(df.isnull().sum())

# Fill missing Monthly_Charges with the median
df["Monthly_Charges"] = df["Monthly_Charges"].fillna(
    df["Monthly_Charges"].median()
)

print("\nMissing values after cleaning:")
print(df.isnull().sum())


## 4. Remove duplicate records

In [ ]:
print("Duplicate rows before:", df.duplicated().sum())

df = df.drop_duplicates().reset_index(drop=True)

print("Duplicate rows after:", df.duplicated().sum())


## 5. Standardize inconsistent text

In [ ]:
# Remove extra spaces and standardize the Contract column
df["Contract"] = df["Contract"].str.strip().str.title()

print(df["Contract"].value_counts())


## 6. Convert columns to correct data types

In [ ]:
df["Monthly_Charges"] = pd.to_numeric(df["Monthly_Charges"])
df["Tenure_Months"] = pd.to_numeric(df["Tenure_Months"], downcast="integer")
df["Support_Calls"] = pd.to_numeric(df["Support_Calls"], downcast="integer")
df["Data_Usage"] = pd.to_numeric(df["Data_Usage"])

print(df.dtypes)


## 7. Handle a simple outlier

In [ ]:
# Support calls should not contain an extremely large value.
# Replace values above 20 with 20.
df["Support_Calls"] = df["Support_Calls"].clip(upper=20)

print("Maximum Support Calls:", df["Support_Calls"].max())


## 8. Feature engineering

In [ ]:
# Create a new feature: estimated yearly charges
df["Yearly_Charges"] = (df["Monthly_Charges"] * 12).round(2)

display(df.head())


## 9. Normalize numerical data

In [ ]:
numeric_columns = [
    "Monthly_Charges",
    "Tenure_Months",
    "Support_Calls",
    "Data_Usage",
    "Yearly_Charges"
]

scaler = MinMaxScaler()
df[numeric_columns] = scaler.fit_transform(df[numeric_columns])

display(df.head())


## 10. Split the data into training and testing sets

In [ ]:
# Separate features and target
X = df.drop("Churn", axis=1)
y = df["Churn"]

# Convert categorical Contract column into numeric columns
X = pd.get_dummies(X, columns=["Contract"], drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data :", X_test.shape)


## 11. Export the cleaned dataset

In [ ]:
df.to_csv("Cleaned_Telecom_Customer_Churn.csv", index=False)

print("Cleaned dataset exported successfully.")
display(df.head())


## Conclusion

The telecom customer data was cleaned and prepared by:
- Handling missing values
- Removing duplicate records
- Standardizing text
- Correcting data types
- Handling a simple outlier
- Creating a new feature
- Normalizing numerical columns
- Splitting the data into training and testing sets
- Exporting the cleaned dataset
